# 17 — Per-Bank Disagreement vs VIX

For each central bank (FED, ECB, BoE), correlates two meeting-level LLM disagreement metrics with the **VIX level** (CBOE Implied Volatility Index) — a forward-looking, options-priced measure of market uncertainty.

VIX is used for all three banks as a common global uncertainty benchmark. It is already an implied volatility measure, so no rolling-window transformation is applied.

| Metric | Description |
|---|---|
| `std_dev_5class` | Mean per-turn ordinal std dev across 6 models (−2…+2 scale) |
| `binary_split_rate` | Fraction of turns where models split on neutral vs directional |

| Timing | Description |
|---|---|
| Contemporaneous | VIX level on (or nearest to) meeting date |
| +1 trading day | VIX level on first trading day after meeting |

Statistics: Pearson r, naive p, N_eff-adjusted p (AR(1) correction), block-bootstrap p + 95% CI.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

ROOT = Path("..")
AGG  = ROOT / "output" / "stance"
CTRL = ROOT / "data" / "controls"
OUT  = AGG

MODELS = [
    "deepseekv3", "gemini25flash", "gpt-4o",
    "llama33", "mistrallarge_or", "qwen25_72b",
]

ORDINAL = {
    "dovish": -2, "mostly dovish": -1, "neutral": 0,
    "mostly hawkish": 1, "hawkish": 2,
}
# 3-class collapse: dovish/mostly dovish → -1, neutral → 0, hawkish/mostly hawkish → +1
THREE_CLASS = {
    "dovish": -1, "mostly dovish": -1, "neutral": 0,
    "mostly hawkish": 1, "hawkish": 1,
}
DIRECTIONAL = {"dovish", "mostly dovish", "mostly hawkish", "hawkish"}

BANKS = {
    "Fed": dict(label="FED", ctrl_file="FED_CONTROLS.csv", color="#E63946", date_fmt="%Y%m%d"),
    # ECB dates in predictions are 8-digit full dates (e.g. 20150122), not YYYYMM
    "ECB": dict(label="ECB", ctrl_file="ECB_CONTROLS.csv", color="#2196F3", date_fmt="%Y%m%d"),
    "BoE": dict(label="BoE", ctrl_file="BOE_CONTROLS.csv", color="#4CAF50", date_fmt="%Y%m"),
}

BLOCK  = 4
N_BOOT = 3000
rng = np.random.default_rng(42)

## 1  Load turn predictions

In [ ]:
frames = []
for m in MODELS:
    df = pd.read_csv(
        AGG / f"turn_predictions_{m}.csv",
        usecols=["bank", "date", "turn_uid", "label"],
    )
    df["model"] = m
    frames.append(df)

all_turns = pd.concat(frames, ignore_index=True)
all_turns["label_lc"]  = all_turns["label"].str.strip().str.lower()
all_turns["ordinal"]   = all_turns["label_lc"].map(ORDINAL)
all_turns["ordinal_3"] = all_turns["label_lc"].map(THREE_CLASS)
all_turns["binary"]    = all_turns["label_lc"].isin(DIRECTIONAL).astype(int)
all_turns = all_turns.dropna(subset=["ordinal"])

print(f"Total turns loaded: {len(all_turns):,}")
print(f"Banks: {all_turns['bank'].unique()}")
print(f"\nLabel distribution:\n{all_turns['label_lc'].value_counts()}")

## 2  Load VIX daily

In [8]:
vix = (
    pd.read_csv(CTRL / "vix_daily.csv", parse_dates=["Date"])
    .rename(columns={"Date": "date_d", "VIX": "vix"})
    .dropna(subset=["vix"])
    .sort_values("date_d")
    .reset_index(drop=True)
)

print(f"VIX daily: {len(vix)} rows | {vix['date_d'].min().date()} to {vix['date_d'].max().date()}")
print(vix.tail(3))

VIX daily: 2854 rows | 2015-01-02 to 2026-05-08
         date_d        vix
2851 2026-05-06  17.389999
2852 2026-05-07  17.080000
2853 2026-05-08  17.190001


## 3  VIX lookup helpers

In [9]:
def vix_contemp(target_date, max_lag=5):
    """VIX level on the nearest trading day within ±max_lag calendar days."""
    window = vix[
        (vix["date_d"] >= target_date - pd.Timedelta(days=max_lag)) &
        (vix["date_d"] <= target_date + pd.Timedelta(days=max_lag))
    ].copy()
    if window.empty:
        return np.nan
    window["diff"] = (window["date_d"] - target_date).abs()
    return window.loc[window["diff"].idxmin(), "vix"]


def vix_next1d(target_date):
    """VIX level on the first trading day strictly after meeting date."""
    future = vix[vix["date_d"] > target_date]
    if future.empty:
        return np.nan
    return future.iloc[0]["vix"]

## 4  Compute per-meeting disagreement metrics and attach VIX

In [ ]:
def build_bank_panel(bank_key, cfg):
    turns = all_turns[all_turns["bank"] == bank_key].copy()

    # parse dates
    turns["date"] = pd.to_datetime(
        turns["date"].astype(str), format=cfg["date_fmt"], errors="coerce"
    )
    turns = turns.dropna(subset=["date"])

    # BoE only: YYYYMM → snap to exact meeting date via controls file
    if cfg["date_fmt"] == "%Y%m":
        ctrl = pd.read_csv(CTRL / cfg["ctrl_file"], parse_dates=["date"])
        ctrl["ym"] = ctrl["date"].dt.to_period("M")
        turns["ym"] = turns["date"].dt.to_period("M")
        ym_to_date = ctrl.set_index("ym")["date"].to_dict()
        turns["date"] = turns["ym"].map(ym_to_date)
        turns = turns.dropna(subset=["date"])

    # per-turn metrics across 6 models
    turn_agg = (
        turns.groupby(["date", "turn_uid"])
        .agg(
            std_dev_5class=("ordinal",   lambda x: x.std(ddof=1)),
            std_dev_3class=("ordinal_3", lambda x: x.std(ddof=1)),
            p_directional= ("binary",    "mean"),
            binary_split=  ("binary",    lambda x: int(x.nunique() > 1)),
        )
        .reset_index()
    )

    # aggregate to meeting level; drop any NaT date keys that survive groupby
    meeting = (
        turn_agg.groupby("date")
        .agg(
            std_dev_5class=   ("std_dev_5class", "mean"),
            std_dev_3class=   ("std_dev_3class", "mean"),
            p_directional=    ("p_directional",  "mean"),
            binary_split_rate=("binary_split",   "mean"),
            n_turns=          ("turn_uid",        "nunique"),
        )
        .reset_index()
        .dropna(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    # temporal transforms (sorted chronologically within bank)
    meeting["std_dev_5class_roll3"] = meeting["std_dev_5class"].rolling(3, min_periods=2).mean()
    meeting["std_dev_5class_diff"]  = meeting["std_dev_5class"].diff()
    meeting["p_directional_roll3"]  = meeting["p_directional"].rolling(3, min_periods=2).mean()
    meeting["p_directional_diff"]   = meeting["p_directional"].diff()

    # attach VIX at 2 timings
    meeting["vix_contemp"] = meeting["date"].apply(vix_contemp)
    meeting["vix_next1d"]  = meeting["date"].apply(vix_next1d)

    n_miss_c = meeting["vix_contemp"].isna().sum()
    n_miss_n = meeting["vix_next1d"].isna().sum()
    d_min = meeting["date"].min()
    d_max = meeting["date"].max()
    d_min_s = d_min.date() if pd.notna(d_min) else "N/A"
    d_max_s = d_max.date() if pd.notna(d_max) else "N/A"
    print(f"{cfg['label']}: {len(meeting)} meetings | {d_min_s} to {d_max_s} | "
          f"VIX missing: contemp={n_miss_c}, next1d={n_miss_n}")
    vix_mean = meeting["vix_contemp"].mean()
    vix_str  = f"{vix_mean:.1f}" if pd.notna(vix_mean) else "N/A"
    print(f"  std5={meeting['std_dev_5class'].mean():.3f}  "
          f"std3={meeting['std_dev_3class'].mean():.3f}  "
          f"p_dir={meeting['p_directional'].mean():.3f}  "
          f"split={meeting['binary_split_rate'].mean():.3f}  "
          f"mean VIX={vix_str}")
    return meeting


panels = {k: build_bank_panel(k, v) for k, v in BANKS.items()}

## 5  Robust correlation helper

In [11]:
def robust_corr(x, y, block=BLOCK, n_boot=N_BOOT, rng=rng):
    """
    Pearson r with:
      - naive p
      - N_eff (AR(1)-based effective sample size) and adjusted p
      - block-bootstrap p and 95% CI
    Returns None if fewer than 5 complete pairs.
    """
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = np.asarray(x)[mask], np.asarray(y)[mask]
    n = len(x)
    if n < 5:
        return None

    r, p_naive = stats.pearsonr(x, y)

    rho_x = float(np.corrcoef(x[:-1], x[1:])[0, 1]) if n > 2 else 0.0
    rho_y = float(np.corrcoef(y[:-1], y[1:])[0, 1]) if n > 2 else 0.0
    denom = 1 + rho_x * rho_y
    n_eff = max(4.0, n * (1 - rho_x * rho_y) / denom if denom != 0 else float(n))

    if abs(r) < 1.0:
        t_adj = r * np.sqrt((n_eff - 2) / (1 - r ** 2))
        p_adj = 2 * (1 - stats.t.cdf(abs(t_adj), df=n_eff - 2))
    else:
        p_adj = 0.0

    boot_rs = []
    for _ in range(n_boot):
        n_blk = int(np.ceil(n / block))
        starts = rng.integers(0, max(1, n - block + 1), n_blk)
        idx_b  = np.concatenate(
            [np.arange(s, min(s + block, n)) for s in starts]
        )[:n]
        try:
            boot_rs.append(stats.pearsonr(x[idx_b], y[idx_b])[0])
        except Exception:
            pass

    boot_rs = np.array(boot_rs)
    ci_lo = float(np.percentile(boot_rs, 2.5))
    ci_hi = float(np.percentile(boot_rs, 97.5))
    p_bb  = float(2 * min(np.mean(boot_rs <= 0), np.mean(boot_rs >= 0)))

    return dict(
        r=r, p_naive=p_naive, n=n, n_eff=n_eff,
        p_adj=p_adj, p_bb=p_bb, ci_lo=ci_lo, ci_hi=ci_hi,
        rho_x=rho_x, rho_y=rho_y,
    )

## 6  Scatter plot helper

In [12]:
def scatter_with_stats(ax, x, y, xlabel, ylabel, title):
    """Scatter + OLS line + robust stats box. Returns the stats dict."""
    res = robust_corr(x, y)
    mask = np.isfinite(x) & np.isfinite(y)
    xc, yc = np.asarray(x)[mask], np.asarray(y)[mask]

    ax.scatter(
        xc, yc, color="#F4A261", alpha=0.7, s=38,
        edgecolors="white", linewidth=0.4, zorder=3,
    )

    if res is not None and len(xc) >= 5:
        m_fit, b_fit = np.polyfit(xc, yc, 1)
        xline = np.linspace(xc.min(), xc.max(), 200)
        ax.plot(xline, m_fit * xline + b_fit, color="#C1121F", lw=1.6, zorder=4)

        txt = (
            f"Pearson r = {res['r']:.3f}\n"
            f"Naive p = {res['p_naive']:.4f}  (n={res['n']})\n"
            f"Adj. p = {res['p_adj']:.4f}  (N eff={res['n_eff']:.0f})\n"
            f"Block-bootstrap p = {res['p_bb']:.4f}\n"
            f"Bootstrap 95% CI\n"
            f"  [{res['ci_lo']:.3f}, {res['ci_hi']:.3f}]"
        )
        ax.text(
            0.04, 0.97, txt, transform=ax.transAxes,
            fontsize=7.5, va="top", family="monospace",
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#BBBBBB", alpha=0.92),
        )
    elif res is None:
        ax.text(0.5, 0.5, "Insufficient data", transform=ax.transAxes,
                ha="center", va="center", fontsize=9, color="gray")

    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(title, fontsize=9.5, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=7)

    return res

## 7  Plot — one figure per bank (2 metrics × 2 timings)

In [ ]:
METRICS = [
    ("std_dev_5class",      "Ordinal std dev (5-class, −2…+2)"),
    ("std_dev_3class",      "Ordinal std dev (3-class, −1…+1)"),
    ("p_directional",       "Avg fraction of models → directional"),
    ("binary_split_rate",   "Fraction of turns with model split\n(neutral vs directional)"),
    ("std_dev_5class_roll3","Rolling mean std dev — k=3 (stock)"),
    ("std_dev_5class_diff", "Δ std dev from prev meeting (flow)"),
    ("p_directional_roll3", "Rolling mean p_directional — k=3 (stock)"),
    ("p_directional_diff",  "Δ p_directional from prev meeting (flow)"),
]

TIMINGS = [
    ("vix_contemp", "Contemporaneous\n(VIX at meeting date)"),
    ("vix_next1d",  "+1 Trading Day\n(VIX day after meeting)"),
]

all_results = []

for bank_key, cfg in BANKS.items():
    meeting = panels[bank_key]
    n_rows, n_cols = len(METRICS), len(TIMINGS)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(11, 4.5 * n_rows),
        constrained_layout=True,
    )
    fig.patch.set_facecolor("white")
    fig.suptitle(
        f"{cfg['label']} — LLM Disagreement vs VIX  (n={len(meeting)} meetings)\n"
        f"Block bootstrap CI, AR(1)-corrected N_eff",
        fontsize=13, fontweight="bold",
    )

    for r_idx, (metric, metric_label) in enumerate(METRICS):
        for c_idx, (vix_col, timing_label) in enumerate(TIMINGS):
            ax = axes[r_idx, c_idx]
            x = meeting[metric].values
            y = meeting[vix_col].values

            res = scatter_with_stats(
                ax, x, y,
                xlabel=metric_label,
                ylabel="VIX" if c_idx == 0 else "",
                title=timing_label,
            )

            if res is not None:
                all_results.append(dict(
                    bank=cfg["label"],
                    metric=metric,
                    timing=timing_label.replace("\n", " "),
                    **{k: res[k] for k in ["r", "p_naive", "n", "n_eff", "p_adj", "p_bb", "ci_lo", "ci_hi"]},
                ))

    out_path = OUT / f"disagreement_vix_by_bank_{cfg['label'].lower()}.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    print(f"Saved → {out_path}")
    plt.show()

## 8  Summary table

In [14]:
summary = pd.DataFrame(all_results)
summary["sig_naive"] = summary["p_naive"] < 0.05
summary["sig_adj"]   = summary["p_adj"]   < 0.05
summary["sig_bb"]    = summary["p_bb"]    < 0.05

display_cols = [
    "bank", "metric", "timing", "n", "n_eff", "r",
    "p_naive", "p_adj", "p_bb", "ci_lo", "ci_hi",
    "sig_naive", "sig_adj", "sig_bb",
]

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 150)

print("=== Summary: LLM Disagreement vs VIX — Pearson r + robust p-values ===")
print(summary[display_cols].to_string(index=False))

=== Summary: LLM Disagreement vs VIX — Pearson r + robust p-values ===
bank            metric                                 timing  n   n_eff       r  p_naive  p_adj   p_bb   ci_lo   ci_hi  sig_naive  sig_adj  sig_bb
 FED    std_dev_5class  Contemporaneous (VIX at meeting date) 73 47.3099  0.4792   0.0000 0.0006 0.0007  0.2838  0.6001       True     True    True
 FED    std_dev_5class +1 Trading Day (VIX day after meeting) 73 47.0781  0.5358   0.0000 0.0001 0.0013  0.2755  0.6811       True     True    True
 FED binary_split_rate  Contemporaneous (VIX at meeting date) 73 62.3596  0.2146   0.0683 0.0930 0.0580 -0.0101  0.3532      False    False   False
 FED binary_split_rate +1 Trading Day (VIX day after meeting) 73 62.2514  0.2438   0.0377 0.0557 0.0753 -0.0222  0.4172       True    False   False
 BoE    std_dev_5class  Contemporaneous (VIX at meeting date) 40 38.5217 -0.0760   0.6412 0.6479 0.6533 -0.3395  0.2168      False    False   False
 BoE    std_dev_5class +1 Trading Day (VI

In [15]:
# Pivot: r values
pivot_r = summary.pivot_table(
    index=["bank", "metric"], columns="timing", values="r", aggfunc="first"
).round(3)
print("=== Pearson r ===")
print(pivot_r.to_string())

# Pivot: block-bootstrap p values
pivot_p = summary.pivot_table(
    index=["bank", "metric"], columns="timing", values="p_bb", aggfunc="first"
).round(4)
print("\n=== Block-bootstrap p ===")
print(pivot_p.to_string())

=== Pearson r ===
timing                  +1 Trading Day (VIX day after meeting)  Contemporaneous (VIX at meeting date)
bank metric                                                                                          
BoE  binary_split_rate                                 -0.2650                                -0.2600
     std_dev_5class                                    -0.0930                                -0.0760
FED  binary_split_rate                                  0.2440                                 0.2150
     std_dev_5class                                     0.5360                                 0.4790

=== Block-bootstrap p ===
timing                  +1 Trading Day (VIX day after meeting)  Contemporaneous (VIX at meeting date)
bank metric                                                                                          
BoE  binary_split_rate                                  0.0100                                 0.0040
     std_dev_5class                  